# Prosperity Round 4 — Counterparty Alpha Mining

**目标**：识别能产生 ≥3-tick 跨日稳定 alpha 的 counterparty 信号 / interaction / 条件，可直接转化为 trader 规则。

**为什么之前没找到**：之前只测了**单 actor、单 role、平均 follow-through**。真正的 alpha 可能在：
1. **Interaction effects**（多 actor 同 tick 同方向 = 强信号）
2. **Conditional signals**（信号 × spread / I_top / volatility regime）
3. **Per-actor inventory tracking**（actor 累计 net flow 预测后续 flow direction）
4. **Cross-product transfer**（VEV_5300 上 Mark 22 → VFE drift）
5. **Adverse selection prediction**（我们 resting order 被谁吃 = 该方向危险）

**输出**：
- 一个 ranked rule table（按 OOS sharpe / hit-rate / SHAP 排序）
- 直接可粘到 `trader_round4_*.py` 的 Python dict

**CV**：3-fold leave-day-out（每个 day 各做一次 test，避免 day-3 单一过拟合）

In [ ]:
# === Cell 1: 安装 + 上传 R4 数据 ===
!pip install -q lightgbm shap scikit-learn pandas numpy

from google.colab import files
import os
if not os.path.exists('prices_round_4_day_1.csv'):
    print('请上传 6 个文件: prices_round_4_day_{1,2,3}.csv + trades_round_4_day_{1,2,3}.csv')
    files.upload()

In [ ]:
# === Cell 2: 加载 + 基础清洗 ===
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

prices = pd.concat([
    pd.read_csv(f'prices_round_4_day_{d}.csv', sep=';').assign(dayfile=d)
    for d in [1, 2, 3]
], ignore_index=True)
trades = pd.concat([
    pd.read_csv(f'trades_round_4_day_{d}.csv', sep=';').assign(dayfile=d)
    for d in [1, 2, 3]
], ignore_index=True)
prices['mid_price'] = pd.to_numeric(prices['mid_price'], errors='coerce')
for col in ['bid_price_1','bid_price_2','bid_price_3','ask_price_1','ask_price_2','ask_price_3',
            'bid_volume_1','bid_volume_2','bid_volume_3','ask_volume_1','ask_volume_2','ask_volume_3']:
    prices[col] = pd.to_numeric(prices[col], errors='coerce')
prices = prices.sort_values(['dayfile','product','timestamp']).reset_index(drop=True)
trades = trades.sort_values(['dayfile','timestamp','symbol']).reset_index(drop=True)

ACTORS = sorted(set(trades['buyer'].dropna().unique()) | set(trades['seller'].dropna().unique()))
print(f'prices rows: {len(prices)}, trades: {len(trades)}, actors: {ACTORS}')
print(f'products: {sorted(prices["product"].unique())}')

In [ ]:
# === Cell 3: 大规模特征工程 ===
# 输出: 每个 (product, day, timestamp) 一行 + 大量特征

def build_features_for_product(prices_p, trades_all, product):
    df = prices_p[prices_p['product']==product].copy().reset_index(drop=True)
    
    # ---- microstructure ----
    bb = df['bid_price_1']; ba = df['ask_price_1']
    bv = df['bid_volume_1'].fillna(0); av = df['ask_volume_1'].fillna(0)
    df['spread'] = ba - bb
    df['microprice'] = ((bb*av + ba*bv) / (bv+av).replace(0,np.nan))
    df['mid_dev_micro'] = df['microprice'] - df['mid_price']
    df['I_top'] = (bv - av) / (bv + av).replace(0, np.nan)
    df['I_top'] = df['I_top'].fillna(0)
    # depth at L2/L3
    bv2 = df['bid_volume_2'].fillna(0); av2 = df['ask_volume_2'].fillna(0)
    bv3 = df['bid_volume_3'].fillna(0); av3 = df['ask_volume_3'].fillna(0)
    df['I_depth'] = ((bv+bv2+bv3) - (av+av2+av3)) / ((bv+bv2+bv3+av+av2+av3).replace(0,np.nan))
    df['I_depth'] = df['I_depth'].fillna(0)
    df['total_depth'] = bv+bv2+bv3+av+av2+av3
    # returns
    for lag in [1,3,5,10,20,50]:
        df[f'ret_{lag}'] = df['mid_price'].diff(lag)
    # rolling vol
    df['rvol_20'] = df['mid_price'].diff().rolling(20).std()
    df['rvol_100'] = df['mid_price'].diff().rolling(100).std()
    
    # ---- counterparty: per-tick + windowed ----
    tp = trades_all[trades_all['symbol']==product].copy()
    for actor in ACTORS:
        tp[f'b_{actor}'] = np.where(tp['buyer']==actor, tp['quantity'], 0)
        tp[f's_{actor}'] = np.where(tp['seller']==actor, tp['quantity'], 0)
    cp_cols = [f'{x}_{a}' for a in ACTORS for x in ['b','s']]
    if len(tp)==0:
        agg = pd.DataFrame({'timestamp': df['timestamp']})
        for c in cp_cols: agg[c] = 0
    else:
        agg = tp.groupby(['dayfile','timestamp'])[cp_cols].sum().reset_index()
    df = df.merge(agg, on=['dayfile','timestamp'], how='left').fillna({c:0 for c in cp_cols})
    
    # rolling windows of net flow per actor (shifted by 1 to avoid leak)
    for W in [5, 20, 50, 200]:
        for actor in ACTORS:
            net = df[f'b_{actor}'] - df[f's_{actor}']
            df[f'nf_{actor}_W{W}'] = net.rolling(W, min_periods=1).sum().shift(1)
    
    # cumulative since session start (per-actor inventory proxy)
    for actor in ACTORS:
        net = df[f'b_{actor}'] - df[f's_{actor}']
        df[f'cum_{actor}'] = net.groupby(df['dayfile']).cumsum().shift(1).fillna(0)
    
    # ---- interaction features (pairs of actors active same window) ----
    # only most-active actor pairs to keep dims manageable
    for W in [5, 20]:
        for a1, a2 in [('Mark 67','Mark 22'), ('Mark 67','Mark 49'), ('Mark 22','Mark 49'),
                        ('Mark 14','Mark 38'), ('Mark 22','Mark 14'), ('Mark 67','Mark 14')]:
            n1 = df[f'nf_{a1}_W{W}'].fillna(0)
            n2 = df[f'nf_{a2}_W{W}'].fillna(0)
            df[f'inter_{a1}_x_{a2}_W{W}'] = n1 * n2
            df[f'agree_{a1}_{a2}_W{W}'] = ((n1 > 0).astype(int) - (n1 < 0).astype(int)) * \
                                          ((n2 > 0).astype(int) - (n2 < 0).astype(int))
    
    # ---- targets at multiple horizons ----
    for k in [1, 5, 10, 25, 50, 100, 200]:
        df[f'y_{k}'] = df['mid_price'].shift(-k) - df['mid_price']
    
    return df

# Build for top products (most counterparty diversity)
PRODUCTS = ['HYDROGEL_PACK', 'VELVETFRUIT_EXTRACT',
            'VEV_4000', 'VEV_4500', 'VEV_5000', 'VEV_5100',
            'VEV_5200', 'VEV_5300']

feats = {p: build_features_for_product(prices, trades, p) for p in PRODUCTS}
for p, df in feats.items():
    print(f'{p}: {df.shape}')

In [ ]:
# === Cell 4: Cross-product features ===
# 加 VFE 当前 mid 偏离 / 各 voucher 的 cumulative_actor 等到每个产品上
vfe = feats['VELVETFRUIT_EXTRACT'][['dayfile','timestamp','mid_price','I_top','spread']].copy()
vfe.columns = ['dayfile','timestamp','vfe_mid','vfe_Itop','vfe_spread']
for p in PRODUCTS:
    if p == 'VELVETFRUIT_EXTRACT': continue
    feats[p] = feats[p].merge(vfe, on=['dayfile','timestamp'], how='left')
    # voucher: intrinsic-residual
    if p.startswith('VEV_'):
        K = int(p.split('_')[1])
        feats[p]['intrinsic'] = (feats[p]['vfe_mid'] - K).clip(lower=0)
        feats[p]['time_premium'] = feats[p]['mid_price'] - feats[p]['intrinsic']
        # session-rolling tp mean (proxy fair anchor)
        feats[p]['tp_roll'] = feats[p].groupby('dayfile')['time_premium'].transform(
            lambda x: x.rolling(200, min_periods=20).mean().shift(1))
        feats[p]['tp_dev'] = feats[p]['time_premium'] - feats[p]['tp_roll']

print('Cross-product features added.')

In [ ]:
# === Cell 5: 训练 LightGBM (原生 NaN 处理，不 dropna features) ===
import lightgbm as lgb
from sklearn.metrics import r2_score

# Diagnostic: NaN profile per product
print("=" * 60)
print("DIAGNOSTIC — NaN profile per product (top 8 NaN columns)")
print("=" * 60)
for p in PRODUCTS:
    df = feats[p]
    nans = df.isna().sum().sort_values(ascending=False)
    nans = nans[nans > 0].head(8)
    print(f"\n{p}: rows={len(df)}, total cols={len(df.columns)}")
    if len(nans) == 0:
        print("  No NaNs.")
    else:
        for col, n in nans.items():
            print(f"  {col:<35s} NaN={n:>6d} ({100*n/len(df):.1f}%)")
print()

def select_features(df):
    drop = {'day','dayfile','timestamp','product','profit_and_loss','mid_price','microprice',
            'bid_price_1','bid_price_2','bid_price_3','ask_price_1','ask_price_2','ask_price_3'}
    drop |= {c for c in df.columns if c.startswith('y_')}
    out = []
    for c in df.columns:
        if c in drop: continue
        if pd.api.types.is_numeric_dtype(df[c]):
            out.append(c)
    return out

def train_loocv(df, horizon, n_iter=400, max_depth=6, lr=0.03):
    """Leave-day-out CV. Only drop NaN on TARGET — LightGBM handles feature NaN natively."""
    fcols = select_features(df)
    target = f'y_{horizon}'
    df_c = df.dropna(subset=[target]).reset_index(drop=True)
    if len(df_c) < 1000: return None
    results = []
    models = {}
    for hold_day in [1, 2, 3]:
        train = df_c[df_c['dayfile'] != hold_day]
        test = df_c[df_c['dayfile'] == hold_day]
        if len(test) < 500: continue
        m = lgb.LGBMRegressor(
            n_estimators=n_iter, learning_rate=lr, max_depth=max_depth,
            num_leaves=31, min_child_samples=200, reg_alpha=0.1, reg_lambda=1.0,
            verbose=-1, random_state=0)
        m.fit(train[fcols], train[target])
        pred_te = m.predict(test[fcols])
        r2 = r2_score(test[target], pred_te)
        signal = np.sign(pred_te)
        pnl = signal * test[target].values
        sharpe = pnl.mean() / (pnl.std() + 1e-9) * np.sqrt(252)
        results.append({'hold_day': hold_day, 'r2': r2, 'sharpe': sharpe,
                        'mean_pnl': pnl.mean(), 'n_test': len(test)})
        models[hold_day] = (m, fcols, train, test, pred_te)
    return results, models

all_results = {}
all_models = {}
print("=" * 60)
print("TRAINING (per product × horizon)")
print("=" * 60)
for p in PRODUCTS:
    n_features = len(select_features(feats[p]))
    print(f"\n=== {p} (features={n_features}, rows={len(feats[p])}) ===")
    for h in [5, 25, 50, 200]:
        r = train_loocv(feats[p], h)
        if r is None:
            print(f"  k={h:>3d}  SKIP (rows<1000 after target dropna)")
            continue
        results, models = r
        avg_r2 = np.mean([x['r2'] for x in results])
        avg_sh = np.mean([x['sharpe'] for x in results])
        avg_pnl = np.mean([x['mean_pnl'] for x in results])
        per_day = ', '.join(f"d{x['hold_day']}r2={x['r2']:+.3f}" for x in results)
        flag = ' ←★' if avg_sh > 0.5 or avg_pnl > 0.2 else ''
        print(f"  k={h:>3d}  avg_r2={avg_r2:+.4f}  avg_sharpe={avg_sh:+.2f}  avg_mean_pnl={avg_pnl:+.4f}  ({per_day}){flag}")
        all_results[(p,h)] = results
        all_models[(p,h)] = models


In [ ]:
# === Cell 6: 全部 (p, h) 排序 + SHAP top features ===
import shap

all_pairs = []
for (p, h), results in all_results.items():
    avg_sh = np.mean([x['sharpe'] for x in results])
    avg_pnl = np.mean([x['mean_pnl'] for x in results])
    avg_r2 = np.mean([x['r2'] for x in results])
    all_pairs.append((p, h, avg_sh, avg_pnl, avg_r2))
all_pairs.sort(key=lambda x: -abs(x[2]))

print("All (p, h) sorted by |sharpe|:")
print(f"{'product':<22s} {'k':>4s} {'sharpe':>8s} {'mean_pnl':>10s} {'avg_r2':>10s}")
for p, h, sh, pnl, r2 in all_pairs:
    flag = ' ←★' if abs(sh) > 0.5 or abs(pnl) > 0.2 else ''
    print(f"{p:<22s} {h:>4d}  {sh:>+8.2f}  {pnl:>+10.4f}  {r2:>+10.4f}{flag}")

# LOOSER filter: anything with non-trivial signal worth examining
promising = [(p, h, sh, pnl) for p, h, sh, pnl, r2 in all_pairs
             if abs(sh) > 0.3 or abs(pnl) > 0.1]
print(f"\nPromising (|sharpe|>0.3 or |mean_pnl|>0.1): {len(promising)}")

def shap_top_features(p, h, top_n=15):
    if (p, h) not in all_models: return
    models = all_models[(p, h)]
    if 3 not in models: return
    m, fcols, train, test, pred = models[3]
    n_samp = min(2000, len(test))
    idx = np.random.RandomState(0).choice(len(test), n_samp, replace=False)
    Xs = test[fcols].iloc[idx]
    expl = shap.TreeExplainer(m)
    sv = expl.shap_values(Xs)
    abs_mean = np.abs(sv).mean(axis=0)
    rank = np.argsort(abs_mean)[::-1][:top_n]
    print(f"\n=== SHAP top-{top_n} for {p} k={h} ===")
    for i in rank:
        print(f"  {fcols[i]:<35s}  |shap|={abs_mean[i]:.5f}  meanshap={sv[:,i].mean():+.5f}")

# SHAP for top-10 promising pairs (or all if <10)
for p, h, sh, pnl in promising[:10]:
    shap_top_features(p, h)


In [ ]:
# === Cell 7: Conditional alpha discovery ===
# 给每个 promising (p, h) 找 top SHAP/importance features，做条件 alpha 表：
# 在 feature 高 / 中 / 低分位下，y_h 的均值和 t-stat。哪个分位 alpha 显著？

def conditional_alpha(p, h, feat_name, q=5):
    df = feats[p].copy()
    df = df.dropna(subset=[feat_name, f'y_{h}']).reset_index(drop=True)
    if len(df) < 100: return
    try:
        df['q'] = pd.qcut(df[feat_name], q=q, labels=False, duplicates='drop')
    except Exception:
        return
    print(f"\n  {p} k={h} feature={feat_name}:")
    print(f"    {'q':>3s} {'d1+2_mean':>10s} {'d1+2_n':>7s} {'d3_mean':>10s} {'d3_n':>7s} {'d3_t':>7s}")
    for q_val in sorted(df['q'].dropna().unique()):
        sub = df[df['q'] == q_val]
        in_s = sub[sub['dayfile'].isin([1, 2])][f'y_{h}']
        out_s = sub[sub['dayfile'] == 3][f'y_{h}']
        if len(out_s) < 20: continue
        t = out_s.mean() * np.sqrt(len(out_s)) / (out_s.std() + 1e-9)
        print(f"    {int(q_val):>3d} {in_s.mean():>+10.3f} {len(in_s):>7d} {out_s.mean():>+10.3f} {len(out_s):>7d} {t:>+7.2f}")

# Iterate top promising (or all if promising is empty fall back to top 10 |sharpe|)
candidates = promising[:5] if len(promising) > 0 else [(p, h, sh, pnl) for p, h, sh, pnl, r2 in all_pairs[:10]]

for p, h, sh, pnl in candidates:
    if (p, h) not in all_models: continue
    models_dict = all_models[(p, h)]
    if 3 not in models_dict: continue
    # FIX: dict[hold_day] gives the 5-tuple (m, fcols, train, test, pred_te)
    m, fcols, _, test, _ = models_dict[3]
    imp = pd.Series(m.feature_importances_, index=fcols).sort_values(ascending=False)
    for fn in imp.head(3).index:
        conditional_alpha(p, h, fn)


In [ ]:
# === Cell 8: Adverse-selection ML (LightGBM 原生 NaN，不 dropna features) ===
from sklearn.metrics import roc_auc_score

for product, smart_actor, smart_role in [
    ('HYDROGEL_PACK', 'Mark 14', 'seller'),
    ('HYDROGEL_PACK', 'Mark 38', 'buyer'),
    ('VELVETFRUIT_EXTRACT', 'Mark 14', 'seller'),
    ('VELVETFRUIT_EXTRACT', 'Mark 67', 'buyer'),
    ('VELVETFRUIT_EXTRACT', 'Mark 22', 'seller'),
]:
    df = feats[product].copy()
    smart_col = f"{'s' if smart_role=='seller' else 'b'}_{smart_actor}"
    if smart_col not in df.columns:
        print(f"\n{product}: column {smart_col} missing, skip")
        continue
    df['target_adverse_1'] = (df.groupby('dayfile')[smart_col].shift(-1) > 0).astype(int)

    fcols = [c for c in select_features(df) if c not in {'target_adverse_1'}]
    print(f"\n=== Adverse: {product}, predict {smart_actor} {smart_role} next 1 tick ===")
    # Only drop NaN on TARGET, let LightGBM handle feature NaN
    df_c = df.dropna(subset=['target_adverse_1']).reset_index(drop=True)
    print(f"  rows after target dropna: {len(df_c)}, target positive rate: {df_c['target_adverse_1'].mean():.4f}")

    avg_aucs = []
    avg_lifts = []
    for hold in [1, 2, 3]:
        train = df_c[df_c['dayfile'] != hold]
        test = df_c[df_c['dayfile'] == hold]
        if len(test) < 500: continue
        if len(train['target_adverse_1'].unique()) < 2:
            print(f"  hold_d{hold}: train has only one class, skip")
            continue
        m = lgb.LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.03,
                                min_child_samples=200, reg_lambda=1.0,
                                verbose=-1, random_state=0,
                                class_weight='balanced')
        m.fit(train[fcols], train['target_adverse_1'])
        if len(test['target_adverse_1'].unique()) < 2:
            print(f"  hold_d{hold}: test has only one class, skip")
            continue
        prob = m.predict_proba(test[fcols])[:, 1]
        auc = roc_auc_score(test['target_adverse_1'], prob)
        df_t = test.copy(); df_t['p'] = prob
        topQ = df_t.sort_values('p', ascending=False).head(int(len(df_t) * 0.05))
        hit_rate = topQ['target_adverse_1'].mean()
        base_rate = test['target_adverse_1'].mean()
        lift = hit_rate / base_rate if base_rate > 0 else 0
        avg_aucs.append(auc); avg_lifts.append(lift)
        print(f"  hold_d{hold}: AUC={auc:.3f}  top-5%-hit={hit_rate:.3f} (base {base_rate:.3f})  lift={lift:.2f}x")
    if avg_aucs:
        print(f"  ★ avg AUC: {np.mean(avg_aucs):.3f}  avg lift: {np.mean(avg_lifts):.2f}x")
    else:
        print(f"  no valid folds (target distribution issue)")


In [ ]:
# === Cell 9 (REPLACEMENT, robust): Distill to ranked rule table ===
# 不再依赖 `promising` 列表 —— 直接遍历所有 (product, horizon) 模型的 top-importance features，
# 加 brute-force 单特征 quantile threshold 扫描，按 |t_oos| 排序。
# 即使 Cell 6 的 promising 是空的，这里也能跑出 rules。

rules = []

# Loop ALL trained models (regardless of sharpe filter)
target_pairs = list(all_models.keys())
# filter to (product, horizon) tuples (skip the adverse-selection entries which have 3 elements)
target_pairs = [k for k in target_pairs if isinstance(k, tuple) and len(k) == 2]

print(f'Scanning {len(target_pairs)} (product, horizon) models...')

for p, h in target_pairs:
    models = all_models[(p, h)]
    # Use day-3-as-test model to get feature importance (most novel test set)
    if 3 not in models:
        continue
    m, fcols, _, _, _ = models[3]
    imp = pd.Series(m.feature_importances_, index=fcols).sort_values(ascending=False)
    # take top 10 importance features (broader scan than top 5)
    for fn in imp.head(10).index:
        df = feats[p].dropna(subset=[fn, f'y_{h}']).reset_index(drop=True)
        if len(df) < 200:
            continue
        # try multiple quantile thresholds
        for q in [0.02, 0.05, 0.10, 0.20, 0.80, 0.90, 0.95, 0.98]:
            try:
                thresh = df[fn].quantile(q)
            except Exception:
                continue
            if q < 0.5:
                mask = df[fn] <= thresh
            else:
                mask = df[fn] >= thresh
            sub = df[mask]
            if len(sub) < 100:
                continue
            in_s = sub[sub['dayfile'].isin([1, 2])][f'y_{h}']
            out_s = sub[sub['dayfile'] == 3][f'y_{h}']
            if len(in_s) < 50 or len(out_s) < 20:
                continue
            in_mean = in_s.mean()
            out_mean = out_s.mean()
            # require sign-consistent
            if np.sign(in_mean) != np.sign(out_mean):
                continue
            out_t = out_mean * np.sqrt(len(out_s)) / (out_s.std() + 1e-9)
            # LOWER threshold from 1.0 → 0.5 to capture more candidates
            if abs(out_t) < 0.5:
                continue
            rules.append({
                'product': p, 'horizon': h, 'feature': fn,
                'condition': '<=' if q < 0.5 else '>=',
                'threshold': float(thresh),
                'in_mean': in_mean, 'out_mean': out_mean,
                'out_t': out_t, 'n_out': len(out_s),
                'fire_rate': len(sub) / len(df),
            })

print(f'\nTotal rule candidates collected: {len(rules)}')

# Empty-handling: ensure DataFrame has expected columns even when empty
columns = ['product', 'horizon', 'feature', 'condition', 'threshold',
           'in_mean', 'out_mean', 'out_t', 'n_out', 'fire_rate']
rules_df = pd.DataFrame(rules, columns=columns)

if len(rules_df) > 0:
    rules_df = rules_df.drop_duplicates(
        subset=['product', 'feature', 'condition', 'threshold', 'horizon']
    )
    rules_df = rules_df.sort_values('out_t', key=lambda s: s.abs(), ascending=False)
    print(f'Sign-consistent + |t_oos|>0.5 rules: {len(rules_df)}')
    print(rules_df.head(40).to_string(
        index=False,
        float_format=lambda x: f'{x:+.3f}' if abs(x) < 1000 else f'{x:.0f}'
    ))
else:
    print('\n*** No rules pass the filters (sign-consistent + |t_oos|>0.5). ***')
    print('This means: ML cross-day alpha really is not present in this data with the current features.')
    print('See Cell 8 (adverse-selection AUC) for the alternate framing.')

rules_df.to_csv('round4_alpha_rules.csv', index=False)
print('\nSaved → round4_alpha_rules.csv')


In [ ]:
# === Cell 10: 生成 trader Python dict (直接粘进 trader_round4_*.py) ===
# 把 rules_df 的 top-N 转成 ENHANCED_RULES = {...} 格式

# Group by product, format as Python dict
TOP_N_PER_PRODUCT = 5
code_lines = ['# === Generated from colab_round4_counterparty.ipynb ===']
code_lines.append('# (paste into trader_round4_*.py as class constants)\n')
code_lines.append('ENHANCED_RULES = {')
for p, grp in rules_df.groupby('product'):
    top = grp.head(TOP_N_PER_PRODUCT)
    code_lines.append(f'    {p!r}: [')
    for _, r in top.iterrows():
        # rule = (feature, condition, threshold, signal_direction)
        sig_dir = +1 if r['out_mean'] > 0 else -1   # +1 = bullish, -1 = bearish
        code_lines.append(
            f'        ({r["feature"]!r}, {r["condition"]!r}, {r["threshold"]:.4f}, {sig_dir}, {r["horizon"]}),  # t_oos={r["out_t"]:+.2f}'
        )
    code_lines.append('    ],')
code_lines.append('}')
code = '\n'.join(code_lines)
print(code)

with open('round4_alpha_rules.py', 'w') as f:
    f.write(code)
print('\nSaved → round4_alpha_rules.py')

# Download for local use
from google.colab import files
files.download('round4_alpha_rules.csv')
files.download('round4_alpha_rules.py')

In [ ]:
# === Cell 11: Ridge regression distillation → linear model weights for trader ===
# 把 LightGBM 的非线性信号蒸馏成可直接 paste 进 trader 的线性模型。
# Per (product, horizon=50/200): 取 top 8 SHAP features → Ridge fit on d1+d2 → 验证 d3 sharpe
# 输出：RIDGE_MODELS dict (features, means, stds, weights, intercept, oos_sharpe)

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

def get_top_features_via_shap(p, h, top_n=8):
    """Return list of feature names with highest |shap| from day-3 model."""
    if (p, h) not in all_models or 3 not in all_models[(p, h)]:
        return None
    m, fcols, _, test, _ = all_models[(p, h)][3]
    n_samp = min(2000, len(test))
    idx = np.random.RandomState(0).choice(len(test), n_samp, replace=False)
    Xs = test[fcols].iloc[idx]
    expl = shap.TreeExplainer(m)
    sv = expl.shap_values(Xs)
    abs_mean = np.abs(sv).mean(axis=0)
    rank = np.argsort(abs_mean)[::-1][:top_n]
    return [fcols[i] for i in rank]

def fit_ridge_for(p, h, top_n=8, ridge_alpha=10.0):
    """Fit Ridge on top-SHAP features. Returns dict with features, means, stds, weights, intercept, sharpe."""
    feats_p = feats[p]
    top_feats = get_top_features_via_shap(p, h, top_n=top_n)
    if top_feats is None: return None

    target = f'y_{h}'
    df = feats_p[top_feats + [target, 'dayfile']].copy()
    df = df.dropna(subset=[target]).reset_index(drop=True)
    # impute remaining NaN with column median (Ridge can't handle NaN)
    for c in top_feats:
        med = df[c].median()
        df[c] = df[c].fillna(med)

    train = df[df['dayfile'].isin([1, 2])]
    test = df[df['dayfile'] == 3]
    if len(train) < 1000 or len(test) < 500:
        return None

    X_tr = train[top_feats].values
    y_tr = train[target].values
    X_te = test[top_feats].values
    y_te = test[target].values

    # Standardize (Ridge prefers scaled features; intercept absorbs scaling)
    means = X_tr.mean(axis=0)
    stds = X_tr.std(axis=0) + 1e-9
    X_tr_z = (X_tr - means) / stds
    X_te_z = (X_te - means) / stds

    m = Ridge(alpha=ridge_alpha)
    m.fit(X_tr_z, y_tr)
    pred_te = m.predict(X_te_z)

    # OOS metrics
    from sklearn.metrics import r2_score
    r2 = r2_score(y_te, pred_te)
    signal = np.sign(pred_te)
    pnl = signal * y_te
    sharpe = pnl.mean() / (pnl.std() + 1e-9) * np.sqrt(252)
    mean_pnl = pnl.mean()

    # Compare with LightGBM sharpe for the same (p, h)
    lgbm_sharpe = np.mean([x['sharpe'] for x in all_results[(p, h)]])
    lgbm_pnl = np.mean([x['mean_pnl'] for x in all_results[(p, h)]])

    return {
        'features': top_feats,
        'means': means.tolist(),
        'stds': stds.tolist(),
        'weights': m.coef_.tolist(),
        'intercept': float(m.intercept_),
        'ridge_oos_r2': float(r2),
        'ridge_oos_sharpe': float(sharpe),
        'ridge_oos_mean_pnl': float(mean_pnl),
        'lgbm_avg_sharpe': float(lgbm_sharpe),
        'lgbm_avg_mean_pnl': float(lgbm_pnl),
        'feature_importance_signed': dict(zip(top_feats, m.coef_.tolist())),
    }

# Fit Ridge for promising (product, horizon) pairs at k=50 and k=200
TARGET_HORIZONS = [50, 200]

ridge_models = {}
print("=" * 70)
print("RIDGE DISTILLATION — comparing OOS sharpe (Ridge vs LightGBM)")
print("=" * 70)
print(f"{'product':<22s} {'k':>4s} {'ridge_sh':>10s} {'lgbm_sh':>10s} {'retain%':>8s} {'r_pnl':>8s} {'lg_pnl':>8s}")

for p in PRODUCTS:
    for h in TARGET_HORIZONS:
        if (p, h) not in all_models: continue
        res = fit_ridge_for(p, h, top_n=8)
        if res is None: continue
        retain = 100 * res['ridge_oos_sharpe'] / res['lgbm_avg_sharpe'] if abs(res['lgbm_avg_sharpe']) > 0.01 else 0
        flag = ' ←' if abs(res['ridge_oos_sharpe']) > 0.5 else ''
        print(f"{p:<22s} {h:>4d}  {res['ridge_oos_sharpe']:>+10.2f} {res['lgbm_avg_sharpe']:>+10.2f} {retain:>+7.0f}% {res['ridge_oos_mean_pnl']:>+8.3f} {res['lgbm_avg_mean_pnl']:>+8.3f}{flag}")
        ridge_models[(p, h)] = res

print(f"\nFitted {len(ridge_models)} Ridge models.")

# Print full feature importance for top 5 (by ridge sharpe)
sorted_models = sorted(ridge_models.items(), key=lambda kv: -abs(kv[1]['ridge_oos_sharpe']))
print("\n" + "=" * 70)
print("TOP RIDGE MODELS — feature importance (signed weight on z-scored feature)")
print("=" * 70)
for (p, h), r in sorted_models[:8]:
    print(f"\n{p} k={h} (ridge sharpe={r['ridge_oos_sharpe']:+.2f}, mean_pnl={r['ridge_oos_mean_pnl']:+.3f})")
    for feat, w in sorted(r['feature_importance_signed'].items(), key=lambda kv: -abs(kv[1])):
        print(f"  {feat:<35s}  weight={w:+.4f}")


In [ ]:
# === Cell 12: Export RIDGE_MODELS as Python dict (paste into trader) ===
# Compact format: trader 里只需要 features 列表 + means + stds + weights + intercept

import json as _json
from google.colab import files

# Filter to models with reasonable OOS sharpe (>0.3 or <-0.3)
deployable = {k: v for k, v in ridge_models.items()
              if abs(v['ridge_oos_sharpe']) >= 0.3}

print(f"Deployable models (|ridge_sharpe|>=0.3): {len(deployable)}")
print()

# Dump as Python literal (ready to paste)
out_lines = ['# === Auto-generated from colab_round4_counterparty.ipynb Cell 12 ===']
out_lines.append('# Use as RIDGE_MODELS class constant in trader_round4_*.py')
out_lines.append('# Predict: drift = intercept + sum(w_i * (feat_i - mean_i) / std_i) for i in features')
out_lines.append('')
out_lines.append('RIDGE_MODELS = {')
for (p, h), r in sorted(deployable.items()):
    sh = r['ridge_oos_sharpe']; pnl = r['ridge_oos_mean_pnl']
    out_lines.append(f'    ({p!r}, {h}): {{   # ridge_oos sharpe={sh:+.2f}  mean_pnl={pnl:+.3f}')
    out_lines.append(f'        "features":  {r["features"]!r},')
    means_s = ', '.join(f'{x:+.6g}' for x in r["means"])
    stds_s  = ', '.join(f'{x:+.6g}' for x in r["stds"])
    weights_s = ', '.join(f'{x:+.6g}' for x in r["weights"])
    out_lines.append(f'        "means":    [{means_s}],')
    out_lines.append(f'        "stds":     [{stds_s}],')
    out_lines.append(f'        "weights":  [{weights_s}],')
    out_lines.append(f'        "intercept": {r["intercept"]:+.6g},')
    out_lines.append('    },')
out_lines.append('}')

code = '\n'.join(out_lines)
print(code[:3000])  # preview first 3000 chars
print('... (truncated for preview)') if len(code) > 3000 else None

with open('round4_ridge_models.py', 'w') as f:
    f.write(code)
with open('round4_ridge_models.json', 'w') as f:
    _json.dump({f"{p}|{h}": v for (p, h), v in deployable.items()}, f, indent=2)
print('\nSaved → round4_ridge_models.py and round4_ridge_models.json')

files.download('round4_ridge_models.py')
files.download('round4_ridge_models.json')


## 跑完后回到本地

1. 把 `round4_alpha_rules.csv` 和 `round4_alpha_rules.py` 下载到本地
2. 把 csv 给我（粘内容到 chat）
3. 我会把规则编进 trader_round4_v4.py 并 backtest 验证

## 如果发现 Cell 5 全部 sharpe 都 < 1

意味着 backtest 数据里确实没有可挖的 cross-day alpha。下一步方向：
1. 跑 Cell 8 的 adverse-selection AUC，看是否有可用的 toxic flow detection
2. 检查我手上的 R4 数据是否完整 — 是否需要用官方私有 prosperity3bt round4 数据